# Chapter 22: Migrating to Actions: Jenkins, GitLab, Importer (notebook edition)

## Learning Objectives

- Convert stage order into needs and predict waves
- Replace Jenkins post with a final if: always() job
- Explain job-level versus step-level failure()
- Price a split pipeline in billed minutes

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Stage order becomes `needs`

In [ ]:
import yaml
from intro_gha.migrate import gitlab_to_workflow
from intro_gha.workflow import execution_waves

doc = yaml.safe_load((ROOT / "sandbox" / "ch22" / "gitlab-ci.yml").read_text())
wf, notes = gitlab_to_workflow(doc)
graph = {j: spec.get("needs", []) for j, spec in wf["jobs"].items()}
assert execution_waves(graph) == [["lint"], ["test"], ["deploy"]]
assert execution_waves({j: [] for j in graph}) == [["deploy", "lint", "test"]]   # stages lost
print(notes)

## 2. `post` has no keyword: read the real runs

In [ ]:
import json
S = json.loads((ROOT / "fixtures" / "migrating_ch22_summary.json").read_text())
f = S["fail_run"]
assert f["jobs"]["deploy"] == "skipped" and f["jobs"]["post"] == "success"
assert f["post_report"]["test"] == "failure"
# job-level failure() ran; step-level failure() in the post job was skipped
assert f["job_level_failure_job"] == "success"
assert f["post_job_step_level_failure_step"] == "skipped"
print(f["post_report"])

## 3. Cost of a split pipeline

In [ ]:
from intro_gha.cost import billed_minutes

stages = [20, 30, 15, 25]
split = sum(billed_minutes(s) for s in stages)
single = billed_minutes(sum(stages))
assert (split, single) == (4, 2)
print("split:", split * 6, "mills; single:", single * 6, "mills")

## Chapter Link

Read: `learning_modules/chapter_22_migrating_to_actions.md`